#### Section C: Question 4:

Use the Transfer learning technique to improve the previous section model’s classification performance.

The pre-trained models weights are given to you. The architecture of pre-trained model till convolution layers and its corresponding weights are already saved under the folder ‘base_model’. The given model convolution layers already freezed. Load these weights along with architecture using the following syntax:

cust_model=tf.keras.models.load_model("base_model") 

“base_model” is the folder name under all the required models files are exist. 

Design the remaining layers of network in your own way (from flattening to output layer) and train only its weights with the dataset given.

In [7]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense, GlobalAveragePooling2D

gpu_devices = tf.config.list_physical_devices('GPU')

if not gpu_devices:
    print("TensorFlow is using the CPU.")
else:
    print(f"TensorFlow is using the following GPU(s): {gpu_devices}")

TensorFlow is using the CPU.


In [8]:
train_dir="3_food_classes/train/"
test_dir="3_food_classes/test/"

In [10]:
# Load the pretrained base model (this model contains only the convolution layers)
base_model=tf.keras.models.load_model("base_model") # "base_model" is the folder given to you with the model files

In [12]:
base_model.summary()

Model: "mobilenetv2_1.00_224"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_3 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 Conv1_pad (ZeroPadding2D)   (None, 225, 225, 3)          0         ['input_3[0][0]']             
                                                                                                  
 Conv1 (Conv2D)              (None, 112, 112, 32)         864       ['Conv1_pad[0][0]']           
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 112, 112, 32)         128       ['Conv1[0][0]']               
 on)                                                                           

In [ ]:
# Freeze the convolutional layers
base_model.trainable = False

# Add custom layers
transfer_model = Sequential([
    base_model,
    Flatten(),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(3, activation='softmax')  # Assuming 3 classes
])

# Compile the model
transfer_model.compile(optimizer='adam',
                       loss='categorical_crossentropy',
                       metrics=['accuracy'])

datagram = ImageDataGenerator(rescale=1./255)

# Input shape changes to 224, 224, 3 as per the base model summary
train_data = datagram.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=20,
    # Because these are not binary classification
    class_mode='categorical'
)

test_data = datagram.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=20,
    # Also align the final output layer as per this
    class_mode='categorical'
)

# Train the model with the dataset
transfer_model.fit(train_data, validation_data=test_data, epochs=10)

# Save the transfer learning model
transfer_model.save("transfer_learning_model.h5")
# Best accuracy we got was 33% - with transfer learing we have 90%!

Found 225 images belonging to 3 classes.
Found 30 images belonging to 3 classes.
Epoch 1/10
12/12 [==============================] - 6s 354ms/step - loss: 3.2670 - accuracy: 0.6356 - val_loss: 0.7334 - val_accuracy: 0.8667
Epoch 2/10
12/12 [==============================] - 4s 294ms/step - loss: 0.5738 - accuracy: 0.9378 - val_loss: 2.3277 - val_accuracy: 0.7333
Epoch 3/10
12/12 [==============================] - 4s 333ms/step - loss: 0.1762 - accuracy: 0.9556 - val_loss: 1.6864 - val_accuracy: 0.8000
Epoch 4/10
12/12 [==============================] - 4s 342ms/step - loss: 0.1381 - accuracy: 0.9778 - val_loss: 1.8925 - val_accuracy: 0.8667
Epoch 5/10
12/12 [==============================] - 4s 321ms/step - loss: 0.1686 - accuracy: 0.9822 - val_loss: 2.0513 - val_accuracy: 0.7667
Epoch 6/10
12/12 [==============================] - 4s 309ms/step - loss: 0.0800 - accuracy: 0.9867 - val_loss: 3.8110 - val_accuracy: 0.7333
Epoch 7/10
12/12 [==============================] - 4s 322ms/step -

/home/vishrut/.pyenv/versions/scrape/lib/python3.10/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
